# Benchmark — Version 0.1 preliminary pilot

This notebook loads `Qwen/Qwen3-VL-2B-Instruct` in float16 at the pinned revision and saves every raw response. It does not score the model and it does not fill ground truth.

The frozen run sends all 60 verified photographs and the three locked prompts. Those verified cells are the OCR ground truth. This notebook does not change them.

1. Colab: **Runtime → Change runtime type → T4 GPU**.
2. On your computer, run `python3 scripts/build_phase0_bundle.py`.
3. Upload this notebook, then upload `dist/phase0_colab_bundle.zip` and `raw.zip` when the cells ask.
4. If the runtime disconnects, download `results/benchmark/benchmark_predictions.jsonl` from the Colab file browser. Completed rows are kept and skipped on the next run.

`straight`, `tilt`, and `occluded` are phone capture conditions. This is not a controlled 3D viewpoint experiment.


In [ ]:
from pathlib import Path
import sys
import zipfile

try:
    from google.colab import files
except ImportError:
    files = None

ROOT = Path('/content/regionalpack-vlm')
ROOT.mkdir(parents=True, exist_ok=True)

def _upload(label):
    if files is None:
        raise RuntimeError('Upload ' + label + ' into ' + str(ROOT) + ' and rerun this cell.')
    print('Upload ' + label)
    uploaded = files.upload()
    if label not in uploaded and uploaded:
        label = next(iter(uploaded))
    dest = ROOT / Path(label).name
    dest.write_bytes(uploaded[label])
    return dest

bundle = ROOT / 'phase0_colab_bundle.zip'
if not (ROOT / 'src' / 'evaluate.py').is_file():
    if not bundle.is_file():
        bundle = _upload('phase0_colab_bundle.zip')
    with zipfile.ZipFile(bundle) as archive:
        archive.extractall(ROOT)

raw_zip = ROOT / 'raw.zip'
if not raw_zip.is_file():
    raw_zip = _upload('raw.zip')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Project root', ROOT)


In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from src.environment import ensure_pillow
print(ensure_pillow(ROOT / 'requirements-colab.txt'))


In [ ]:
from src.integrate import integrate_raw_zip
from src.evaluate import evaluation_block_reason, image_jobs, prompt_jobs

rows = integrate_raw_zip(ROOT, ROOT / 'raw.zip')
print(len(rows), 'images,', sum(row['readable'] == 'yes' for row in rows), 'readable')
print(len(image_jobs(ROOT)), 'image jobs;', len(prompt_jobs(ROOT)), 'prompts each')
reason = evaluation_block_reason(ROOT)
if reason:
    raise RuntimeError(reason)
print('60 verified photographs are ready. The next cell loads the model.')


In [ ]:
from src.evaluate import run_benchmark
paths = run_benchmark(ROOT)
print('Wrote', paths['zip'])
try:
    from google.colab import files
    files.download(str(paths['zip']))
except ImportError:
    pass
